<a href="https://colab.research.google.com/github/AngurisSummer/Informatik-Bullet-Time/blob/main/Hintergrundsentferner_ohne_Greenscreen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Hello World")

Hello World


In [ ]:
pip install "rembg[cpu]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 17.4 MB/s eta 0:00:00


In [10]:
from pathlib import Path
import time

import cv2
import numpy as np
from PIL import Image
from rembg import remove, new_session


# ==========================================================
# SETTINGS
# ==========================================================

INPUT_DIR = Path("input")
OUTPUT_DIR = Path("output")

INPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

SUPPORTED_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".jfif"
}

# Maximum size used for AI processing.
# Larger = better detail, slower.
MAX_SIZE = 1600

# Human-specific segmentation model.
MODEL_NAME = "u2net_human_seg"


# ==========================================================
# LOAD MODEL ONCE
# ==========================================================

print("Loading human segmentation model...")

session = new_session(MODEL_NAME)

print("Model loaded.")


# ==========================================================
# RESIZE IMAGE FOR FASTER PROCESSING
# ==========================================================

def resize_for_processing(image):
    """
    Resize large images before AI inference.

    Keeps aspect ratio.
    Images smaller than MAX_SIZE are unchanged.
    """

    width, height = image.size

    largest_dimension = max(
        width,
        height
    )

    if largest_dimension <= MAX_SIZE:
        return image

    scale = (
        MAX_SIZE
        / largest_dimension
    )

    new_width = int(
        width * scale
    )

    new_height = int(
        height * scale
    )

    resized = image.resize(
        (
            new_width,
            new_height
        ),
        Image.Resampling.LANCZOS
    )

    return resized


# ==========================================================
# CLEAN SEGMENTATION MASK
# ==========================================================

def clean_mask(alpha):
    """
    Improve the AI-generated alpha mask.

    Main idea:

    1. Convert mask temporarily to black/white.
    2. Fill very small gaps.
    3. Find all disconnected foreground regions.
    4. Keep only the largest connected region.
    5. Restore the AI model's soft alpha edges.
    6. Smooth the final edge slightly.

    This works well when the image is supposed to
    contain one main Bullet-Time subject.
    """

    # ------------------------------------------------------
    # 1. Convert alpha mask into binary mask
    # ------------------------------------------------------

    MASK_THRESHOLD = 40

    binary = np.where(
        alpha > MASK_THRESHOLD,
        255,
        0
    ).astype(
        np.uint8
    )


    # ------------------------------------------------------
    # 2. Fill tiny gaps in the detected person
    # ------------------------------------------------------

    kernel = np.ones(
        (3, 3),
        dtype=np.uint8
    )

    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=2
    )


    # ------------------------------------------------------
    # 3. Find disconnected foreground regions
    # ------------------------------------------------------

    (
        number_labels,
        labels,
        stats,
        _
    ) = cv2.connectedComponentsWithStats(
        binary,
        connectivity=8
    )


    # ------------------------------------------------------
    # If nothing was detected, return original alpha
    # ------------------------------------------------------

    if number_labels <= 1:

        print(
            "Warning: no clear foreground detected."
        )

        return alpha


    # ------------------------------------------------------
    # 4. Find largest connected region
    #
    # Label 0 is background, so ignore it.
    # ------------------------------------------------------

    areas = stats[
        1:,
        cv2.CC_STAT_AREA
    ]

    largest_label = (
        1
        + np.argmax(
            areas
        )
    )


    largest_area = stats[
        largest_label,
        cv2.CC_STAT_AREA
    ]

    print(
        f"Main detected region: "
        f"{largest_area} pixels"
    )


    # ------------------------------------------------------
    # 5. Keep only largest region
    # ------------------------------------------------------

    main_person = np.zeros_like(
        binary
    )

    main_person[
        labels == largest_label
    ] = 255


    # ------------------------------------------------------
    # 6. Expand slightly
    #
    # This allows us to recover soft edges such as:
    #
    # - hair
    # - fingers
    # - clothing edges
    # ------------------------------------------------------

    recovery_kernel = np.ones(
        (5, 5),
        dtype=np.uint8
    )

    recovery_region = cv2.dilate(
        main_person,
        recovery_kernel,
        iterations=1
    )


    # ------------------------------------------------------
    # 7. Restore original AI alpha values
    #
    # Keep AI softness inside valid person region.
    # Everything outside becomes transparent.
    # ------------------------------------------------------

    cleaned = np.where(
        recovery_region > 0,
        alpha,
        0
    ).astype(
        np.uint8
    )


    # ------------------------------------------------------
    # 8. Slight edge smoothing
    # ------------------------------------------------------

    cleaned = cv2.GaussianBlur(
        cleaned,
        (3, 3),
        0
    )


    return cleaned


# ==========================================================
# WHITE BACKGROUND PREVIEW
# ==========================================================

def add_white_background(foreground):
    """
    Put transparent subject on a white background.

    This is only for easier visual inspection.
    """

    foreground = foreground.convert(
        "RGBA"
    )

    background = Image.new(
        "RGBA",
        foreground.size,
        (
            255,
            255,
            255,
            255
        )
    )

    combined = Image.alpha_composite(
        background,
        foreground
    )

    return combined.convert(
        "RGB"
    )


# ==========================================================
# REMOVE BACKGROUND
# ==========================================================

def remove_background(image):
    """
    Run human segmentation and clean the resulting mask.
    """

    original_size = image.size


    # ------------------------------------------------------
    # Resize large image for AI processing
    # ------------------------------------------------------

    working_image = resize_for_processing(
        image
    )


    print(
        f"AI processing resolution: "
        f"{working_image.size[0]} x "
        f"{working_image.size[1]}"
    )


    # ------------------------------------------------------
    # Run human segmentation model
    # ------------------------------------------------------

    result = remove(
        working_image,
        session=session,
        alpha_matting=False
    )

    result = result.convert(
        "RGBA"
    )


    # ------------------------------------------------------
    # Convert image to NumPy array
    # ------------------------------------------------------

    array = np.array(
        result
    )


    # ------------------------------------------------------
    # Extract alpha channel
    # ------------------------------------------------------

    alpha = array[
        :,
        :,
        3
    ]


    # ------------------------------------------------------
    # Clean segmentation mask
    # ------------------------------------------------------

    alpha = clean_mask(
        alpha
    )


    # ------------------------------------------------------
    # Put cleaned alpha back into image
    # ------------------------------------------------------

    array[
        :,
        :,
        3
    ] = alpha


    # ------------------------------------------------------
    # Convert back to PIL image
    # ------------------------------------------------------

    result = Image.fromarray(
        array
    )


    # ------------------------------------------------------
    # Resize back to original dimensions
    # ------------------------------------------------------

    if result.size != original_size:

        result = result.resize(
            original_size,
            Image.Resampling.LANCZOS
        )


    return result


# ==========================================================
# PROCESS ONE IMAGE
# ==========================================================

def process_image(image_path):

    print()
    print(
        "======================================="
    )

    print(
        f"Processing: {image_path.name}"
    )

    print(
        "======================================="
    )


    try:

        # --------------------------------------------------
        # Load image
        # --------------------------------------------------

        original = Image.open(
            image_path
        ).convert(
            "RGB"
        )


        print(
            f"Original resolution: "
            f"{original.size[0]} x "
            f"{original.size[1]}"
        )


        # --------------------------------------------------
        # Start timer
        # --------------------------------------------------

        start_time = time.time()


        # --------------------------------------------------
        # Run segmentation
        # --------------------------------------------------

        foreground = remove_background(
            original
        )


        # --------------------------------------------------
        # Stop timer
        # --------------------------------------------------

        elapsed = (
            time.time()
            - start_time
        )


        print(
            f"Segmentation time: "
            f"{elapsed:.2f} seconds"
        )


        # ==================================================
        # SAVE TRANSPARENT PNG
        # ==================================================

        transparent_path = (
            OUTPUT_DIR
            /
            f"{image_path.stem}_transparent.png"
        )


        foreground.save(
            transparent_path
        )


        # ==================================================
        # SAVE WHITE BACKGROUND PREVIEW
        # ==================================================

        white_image = add_white_background(
            foreground
        )


        white_path = (
            OUTPUT_DIR
            /
            f"{image_path.stem}_white.jpg"
        )


        white_image.save(
            white_path,
            quality=95
        )


        # ==================================================
        # SAVE MASK
        # ==================================================

        mask = foreground.getchannel(
            "A"
        )


        mask_path = (
            OUTPUT_DIR
            /
            f"{image_path.stem}_mask.png"
        )


        mask.save(
            mask_path
        )


        # ==================================================
        # PRINT RESULTS
        # ==================================================

        print()
        print("Saved:")

        print(
            f"  {transparent_path}"
        )

        print(
            f"  {white_path}"
        )

        print(
            f"  {mask_path}"
        )


    except Exception as error:

        print()
        print(
            f"ERROR processing "
            f"{image_path.name}"
        )

        print(
            error
        )


# ==========================================================
# MAIN PROGRAM
# ==========================================================

def main():

    print()
    print(
        "======================================="
    )

    print(
        "Bullet-Time Human Background Removal"
    )

    print(
        "======================================="
    )


    # ------------------------------------------------------
    # Find images in input folder
    # ------------------------------------------------------

    images = sorted(
        [
            path
            for path
            in INPUT_DIR.iterdir()

            if path.suffix.lower()
            in SUPPORTED_EXTENSIONS
        ]
    )


    # ------------------------------------------------------
    # Check for empty folder
    # ------------------------------------------------------

    if not images:

        print()
        print(
            "No images found."
        )

        print()
        print(
            "Place JPG/PNG/WebP images inside:"
        )

        print(
            INPUT_DIR.resolve()
        )

        return


    print()
    print(
        f"Found {len(images)} image(s)."
    )


    # ------------------------------------------------------
    # Time complete batch
    # ------------------------------------------------------

    total_start = time.time()


    # ------------------------------------------------------
    # Process every image
    # ------------------------------------------------------

    for image_path in images:

        process_image(
            image_path
        )


    # ------------------------------------------------------
    # Total processing time
    # ------------------------------------------------------

    total_elapsed = (
        time.time()
        - total_start
    )


    print()
    print(
        "======================================="
    )

    print(
        "FINISHED"
    )

    print(
        "======================================="
    )


    print()
    print(
        f"Processed "
        f"{len(images)} image(s)"
    )

    print(
        f"Total time: "
        f"{total_elapsed:.2f} seconds"
    )


    print(
        f"Average time/image: "
        f"{total_elapsed / len(images):.2f} seconds"
    )


    print()
    print(
        "Results saved in:"
    )

    print(
        OUTPUT_DIR.resolve()
    )


# ==========================================================
# START PROGRAM
# ==========================================================

if __name__ == "__main__":

    main()

Loading human segmentation model...
Model loaded.

Bullet-Time Human Background Removal

Found 8 image(s).

Processing: 01_LINEAR_500ms__Kamera_01_50d4d0b9.jpg
Original resolution: 1920 x 1080
AI processing resolution: 1600 x 900
Main detected region: 198945 pixels
Segmentation time: 2.40 seconds

Saved:
  output/01_LINEAR_500ms__Kamera_01_50d4d0b9_transparent.png
  output/01_LINEAR_500ms__Kamera_01_50d4d0b9_white.jpg
  output/01_LINEAR_500ms__Kamera_01_50d4d0b9_mask.png

Processing: 01_LINEAR_500ms__Kamera_02_21b1fe1b.jpg
Original resolution: 1920 x 1080
AI processing resolution: 1600 x 900
Main detected region: 247446 pixels
Segmentation time: 2.45 seconds

Saved:
  output/01_LINEAR_500ms__Kamera_02_21b1fe1b_transparent.png
  output/01_LINEAR_500ms__Kamera_02_21b1fe1b_white.jpg
  output/01_LINEAR_500ms__Kamera_02_21b1fe1b_mask.png

Processing: 01_LINEAR_500ms__Kamera_03_c942442b.jpg
Original resolution: 1920 x 1080
AI processing resolution: 1600 x 900
Main detected region: 155598 pix